---
title: Data-Centric AI (Module II)
subtitle: "Data Augmentation, SMOTE, and Leakage-Safe Pipelines"
---

## Learning outcomes

By the end of this notebook, you should be able to:

- Build `sklearn` pipelines and optimize their hyper-parameters.
- Explain class imbalance and why accuracy can be misleading.
- Distinguish data augmentation from resampling.
- Compare random oversampling, undersampling, SMOTE, ADASYN, and class weighting.
- Place resampling correctly inside a cross-validation pipeline.
- Select appropriate metrics and decision thresholds.


## 1. Why pipelines matter

A machine-learning pipeline is rarely just a classifier. A real workflow often includes missing-value handling, scaling, categorical encoding, feature selection, resampling, and then the final estimator.

`sklearn.pipeline.Pipeline` packages these steps into one object with a single `fit`, `predict`, and `predict_proba` interface.

This matters because every preprocessing step must be learned only from the training data. A pipeline makes that discipline automatic during train/test splits and cross-validation.


## 2. A basic sklearn pipeline

A typical pipeline chains transformers and a final estimator:

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000)),
])

pipeline.fit(X_train, y_train)
pipeline.predict(X_test)
```

The important point is that `StandardScaler` is fitted on `X_train` only. The test set is transformed using the training-set scaling parameters.


## 3. Pipelines expose hyper-parameters

Pipeline parameters are addressed with the pattern:

```text
step_name__parameter_name
```

For example, if the classifier step is named `classifier`, then the logistic-regression regularization strength is `classifier__C`.

```python
param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10],
    "classifier__class_weight": [None, "balanced"],
}
```

This lets us tune preprocessing choices and model choices together, while keeping the evaluation leakage-safe.


## 4. Hyper-parameter optimization with cross-validation

`GridSearchCV` trains and evaluates many pipeline configurations using cross-validation. Each fold fits preprocessing and the classifier only on the training part of that fold.

```python
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
)

search.fit(X_train, y_train)
search.best_params_
```

For imbalanced classification, the scoring function should match the task. Accuracy is often the wrong objective; recall, F1, F-beta, average precision, or a custom cost function may be more appropriate.


## 5. Pipelines with resampling need imblearn

`sklearn.pipeline.Pipeline` is designed for transformers and estimators. Resamplers such as SMOTE change both `X` and `y`, so they need `imblearn.pipeline.Pipeline`.

```python
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

pipeline = ImbPipeline([
    ("preprocessing", preprocessor),
    ("sampler", SMOTE(random_state=42)),
    ("classifier", LogisticRegression(max_iter=2000)),
])
```

The central rule of this lab follows from this: resampling is part of training, so it must happen inside the pipeline and inside each cross-validation fold.


## 6. The imbalanced classification problem

Many important binary classification problems have one class that is much rarer than the other: fraud detection, medical diagnosis, safety incidents, customer churn, credit default, and equipment failures.

We call the frequent class the **majority class** and the rare class the **minority class**. In this notebook the minority class is the positive class: `1`.

When classes are imbalanced, **accuracy can be misleading**. If only 5% of customers churn, a classifier that always predicts "no churn" reaches 95% accuracy while finding no churners at all.

Useful metrics include:

- **Precision**: among predicted positives, how many were actually positive?
- **Recall** or sensitivity: among actual positives, how many did we find?
- **Specificity**: among actual negatives, how many did we correctly reject?
- **F1 score**: harmonic mean of precision and recall.
- **Confusion matrix**: counts of true negatives, false positives, false negatives, and true positives.


## 7. Methods for handling imbalance

Common strategies include:

- **No treatment**: train directly and evaluate carefully.
- **Class weights**: make minority-class errors more expensive.
- **Undersampling**: remove majority examples.
- **Oversampling**: duplicate minority examples.
- **SMOTE and variants**: synthesize minority examples.

Use this slide as the menu of interventions. The rest of the lab asks which intervention changes errors in the desired direction.


## Data augmentation versus SMOTE

Data augmentation applies domain-specific label-preserving transformations: image rotations, sensor noise, audio crops, text perturbations, or simulated operating conditions.

SMOTE is narrower. It creates tabular rows by interpolation in feature space.

The important question is whether the generated record could exist in the real data-generating process.


## ADASYN in practice

ADASYN creates more synthetic minority examples around locally difficult observations: minority points surrounded by many majority neighbors.

This can improve recall in hard regions, but it can also amplify mislabeled points or outliers.

Test it inside an `imblearn` pipeline, using stratified cross-validation, precision, recall, F1, confusion matrices, and threshold-sensitive cost.


## 8. How SMOTE works

SMOTE creates new minority-class observations instead of duplicating existing ones [@chawla2002smote].

For a minority point `x_i`, it selects a nearby minority point `x_j` and creates:

```text
x_new = x_i + lambda * (x_j - x_i)
```

The intuition is simple: fill sparse minority regions. The risk is also simple: interpolation may create impossible records.


## Borderline-SMOTE

Borderline-SMOTE focuses on minority observations near the class boundary [@han2005borderline].

These are the points a classifier is most likely to confuse with the majority class.

This can help boundary recall, but noisy or mislabeled boundary points can be reinforced.


## SMOTE-NC

SMOTE-NC handles mixed numerical and categorical data [@chawla2002smote].

- Numerical columns are interpolated.
- Categorical columns are chosen from neighboring minority examples.

It avoids fractional one-hot categories, but it still cannot guarantee every generated category combination is realistic.


## ADASYN

ADASYN estimates local difficulty from nearest neighbors [@he2008adasyn].

Minority observations surrounded by many majority observations receive larger sampling weights.

The method shifts attention toward hard minority regions, which can be useful or risky depending on label noise and class overlap.


## 9. Leakage warning

Resampling is part of model training. It must be learned from the training data only.

Do **not** apply SMOTE or oversampling before splitting the data. If synthetic points are created before a train/test split, information from validation or test observations can leak into the training set through nearest-neighbor interpolation or duplicated examples.

The same rule applies to cross-validation: resampling must happen **inside each training fold**, not once globally before cross-validation.

This is why we use `imblearn.pipeline.Pipeline` whenever the pipeline contains a sampler.


## 10. Setup

If `imbalanced-learn` is missing in your environment, install it first with:

```bash
pip install imbalanced-learn
```


## Code: Colab dependency setup

Run this cell first in Google Colab. It installs `imbalanced-learn` only when it is missing.

This setup cell keeps the notebook portable: local environments that already have the package skip installation, while Colab installs the missing dependency.


In [ ]:
#| echo: false
# Install the sampler library only when the runtime does not already provide it.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("imblearn") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "imbalanced-learn"])


## Code: Imports and settings

The imports separate standard `sklearn` pipeline pieces from `imblearn` samplers, which is important because samplers need the `imblearn` pipeline implementation.


In [ ]:
#| echo: false
# Keep one seed so every plot and cross-validation split is reproducible in class.
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    accuracy_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import ADASYN, BorderlineSMOTE, RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")


## 11. Create a deliberately imbalanced dataset

We will use a synthetic binary classification dataset so the notebook is reproducible without external data downloads.

Think of `y=1` as a rare high-risk event, such as fraud, churn, default, or failure.


## Code: Generate the dataset

The class weights make the positive class rare but not impossible, so students can observe both false negatives and false positives in later metrics.


In [ ]:
#| echo: false
# Build a controlled rare-event problem with about 6% positive examples.
X_array, y_array = make_classification(
    n_samples=5000,
    n_features=12,
    n_informative=5,
    n_redundant=3,
    n_clusters_per_class=2,
    weights=[0.94, 0.06],
    class_sep=0.9,
    flip_y=0.02,
    random_state=RANDOM_STATE,
)

feature_names = [f"feature_{i:02d}" for i in range(X_array.shape[1])]
X = pd.DataFrame(X_array, columns=feature_names)
y = pd.Series(y_array, name="high_risk")

class_counts = y.value_counts().sort_index()
class_rates = y.value_counts(normalize=True).sort_index()

pd.DataFrame({"count": class_counts, "rate": class_rates.round(3)})


## Code: Plot class balance

Start with the base rate. Every later metric should be interpreted relative to this imbalance.


In [ ]:
#| echo: false
# Visualize the base rate before discussing any model metrics.
fig, ax = plt.subplots(figsize=(5, 3))
class_counts.plot(kind="bar", ax=ax, color=["#4C78A8", "#F58518"])
ax.set_title("Class distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Number of observations")
ax.set_xticklabels(["0: majority", "1: minority"], rotation=0)
plt.show()


## 12. Why accuracy fails

A classifier that always predicts the majority class can look strong by accuracy alone. But it has zero recall for the minority class.


## Code: Majority-class baseline

This deliberately weak baseline is useful because it exposes why accuracy alone is not a sufficient objective.


In [ ]:
#| echo: false
# Predicting only the majority class gives high accuracy but no minority recall.
always_majority = np.zeros_like(y)

accuracy = (always_majority == y).mean()
precision = precision_score(y, always_majority, zero_division=0)
recall = recall_score(y, always_majority, zero_division=0)
f1 = f1_score(y, always_majority, zero_division=0)

pd.DataFrame(
    [{"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}]
).round(3)


## 13. Visual intuition

Before full pipelines, inspect resampling on a small 2D dataset.

The goal is not the best model. The goal is to see how the training set changes.


## What the resamplers change

- Undersampling removes majority rows.
- Oversampling duplicates minority rows.
- SMOTE interpolates between minority neighbors.
- Borderline-SMOTE focuses near the class boundary.
- ADASYN focuses on locally difficult minority regions.

Class weighting is not plotted because it changes the objective, not the rows.


## Code: Build a 2D imbalanced dataset

The 2D dataset is only for visualization. The full experiment later uses all features and cross-validation.


In [ ]:
#| echo: false
# Create a small two-feature dataset so synthetic samples can be plotted directly.
X_2d_array, y_2d_array = make_classification(
    n_samples=350,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    weights=[0.9, 0.1],
    class_sep=0.85,
    flip_y=0.03,
    random_state=RANDOM_STATE,
)

X_2d = pd.DataFrame(X_2d_array, columns=["x1", "x2"])
y_2d = pd.Series(y_2d_array, name="minority_class")

pd.DataFrame({"count": y_2d.value_counts().sort_index()})


## Code: Plot resampling effects in feature space

Each panel shows the training data after one resampling strategy. The validation data is not involved in this illustration.


In [ ]:
#| echo: false
# Apply each sampler to the same toy data and plot the resulting training geometry.
toy_samplers = {
    "Original data": None,
    "Random undersampling": RandomUnderSampler(random_state=RANDOM_STATE),
    "Random oversampling": RandomOverSampler(random_state=RANDOM_STATE),
    "SMOTE": SMOTE(random_state=RANDOM_STATE, k_neighbors=5),
    "Borderline-SMOTE": BorderlineSMOTE(random_state=RANDOM_STATE, k_neighbors=5),
    "ADASYN": ADASYN(random_state=RANDOM_STATE, n_neighbors=5),
}


def apply_sampler_for_plot(sampler, X, y):
    if sampler is None:
        X_resampled = X.copy()
        y_resampled = y.copy()
    else:
        X_values, y_values = sampler.fit_resample(X, y)
        X_resampled = pd.DataFrame(X_values, columns=X.columns)
        y_resampled = pd.Series(y_values, name=y.name)

    original_mask = np.arange(len(X_resampled)) < len(X)
    return X_resampled, y_resampled, original_mask


def plot_resampled_dataset(ax, title, X_resampled, y_resampled, original_mask):
    majority = y_resampled == 0
    minority = y_resampled == 1
    synthetic = ~original_mask

    ax.scatter(
        X_resampled.loc[majority, "x1"],
        X_resampled.loc[majority, "x2"],
        s=22,
        alpha=0.35,
        label="majority",
        color="#4C78A8",
    )
    ax.scatter(
        X_resampled.loc[minority & original_mask, "x1"],
        X_resampled.loc[minority & original_mask, "x2"],
        s=32,
        alpha=0.85,
        label="original minority",
        color="#F58518",
        edgecolor="black",
        linewidth=0.3,
    )
    ax.scatter(
        X_resampled.loc[synthetic, "x1"],
        X_resampled.loc[synthetic, "x2"],
        s=38,
        alpha=0.85,
        label="new or duplicated",
        color="#E45756",
        marker="x",
    )

    counts = y_resampled.value_counts().sort_index()
    ax.set_title(f"{title}\nclass 0: {counts.get(0, 0)}, class 1: {counts.get(1, 0)}")
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")


fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True)

for ax, (name, sampler) in zip(axes.ravel(), toy_samplers.items()):
    X_resampled, y_resampled, original_mask = apply_sampler_for_plot(sampler, X_2d, y_2d)
    plot_resampled_dataset(ax, name, X_resampled, y_resampled, original_mask)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3)
fig.suptitle("How resampling changes a 2D training set", y=1.02, fontsize=16)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()


## Reading the visual comparison

The plots show why these techniques are not interchangeable.

- Random oversampling increases the minority count by repeating existing observations, so the red crosses sit directly on top of known minority points.
- SMOTE fills line segments between nearby minority points. This can smooth sparse minority regions, but it assumes that interpolation produces valid records.
- Borderline-SMOTE tries to synthesize near the frontier between classes. It is useful when boundary recall matters, but it can be sensitive to overlap and label noise.
- ADASYN places more synthetic examples in locally difficult regions. Compared with SMOTE, it often concentrates more strongly near areas where the majority class surrounds the minority class.

The visual lesson carries over to tabular data: synthetic rows are helpful only if the generated feature combinations are plausible for the domain.


## 14. Baseline pipeline

A conventional `sklearn.pipeline.Pipeline` is sufficient when all steps are transformers and estimators.

```python
Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression())
])
```

When we add a sampler such as SMOTE, random oversampling, or undersampling, we switch to `imblearn.pipeline.Pipeline`.


## Code: Build the baseline pipeline

The baseline establishes the ordinary preprocessing-and-classifier structure before samplers are introduced.


In [ ]:
#| echo: false
# Standardize numeric features before logistic regression.
numeric_features = X.columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[("num", StandardScaler(), numeric_features)],
    remainder="drop",
)

baseline_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]
)

baseline_pipeline


## 15. Candidate pipelines

We compare seven strategies:

1. No imbalance treatment.
2. `class_weight="balanced"`.
3. Random undersampling.
4. Random oversampling.
5. SMOTE.
6. ADASYN.
7. Borderline-SMOTE.

All samplers are placed after preprocessing and before the classifier, so they are fitted only on the current training fold.


## Code: Define candidate pipelines

The sampler position is the key idea: preprocessing first, resampling second, classifier last.


In [ ]:
#| echo: false
# Compare ordinary sklearn pipelines with imblearn pipelines that can contain samplers.
def logistic_regression(class_weight=None):
    return LogisticRegression(
        max_iter=2000,
        class_weight=class_weight,
        random_state=RANDOM_STATE,
    )


pipelines = {
    "No treatment": Pipeline(
        [("preprocessing", preprocessor), ("classifier", logistic_regression())]
    ),
    "Class weight": Pipeline(
        [
            ("preprocessing", preprocessor),
            ("classifier", logistic_regression(class_weight="balanced")),
        ]
    ),
    "Random undersampling": ImbPipeline(
        [
            ("preprocessing", preprocessor),
            ("sampler", RandomUnderSampler(random_state=RANDOM_STATE)),
            ("classifier", logistic_regression()),
        ]
    ),
    "Random oversampling": ImbPipeline(
        [
            ("preprocessing", preprocessor),
            ("sampler", RandomOverSampler(random_state=RANDOM_STATE)),
            ("classifier", logistic_regression()),
        ]
    ),
    "SMOTE": ImbPipeline(
        [
            ("preprocessing", preprocessor),
            ("sampler", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
            ("classifier", logistic_regression()),
        ]
    ),
    "ADASYN": ImbPipeline(
        [
            ("preprocessing", preprocessor),
            ("sampler", ADASYN(random_state=RANDOM_STATE, n_neighbors=5)),
            ("classifier", logistic_regression()),
        ]
    ),
    "Borderline-SMOTE": ImbPipeline(
        [
            ("preprocessing", preprocessor),
            ("sampler", BorderlineSMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
            ("classifier", logistic_regression()),
        ]
    ),
}

list(pipelines)


## 16. Stratified cross-validation

The helper below manually loops over stratified folds. This makes it explicit that every pipeline is cloned, fitted on the training fold, and evaluated on the untouched validation fold.


## Code: Run stratified cross-validation

The explicit loop makes leakage checks visible: every fold gets a fresh clone of the pipeline.


In [ ]:
#| echo: false
# Fit a fresh pipeline in each fold so preprocessing and sampling never see validation rows.
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fp) if (tn + fp) else np.nan


def evaluate_pipeline(name, pipeline, X, y, cv, threshold=0.5):
    fold_rows = []
    out_of_fold_scores = pd.Series(index=y.index, dtype=float)

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        model = clone(pipeline)
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        model.fit(X_train, y_train)
        y_score = model.predict_proba(X_valid)[:, 1]
        y_pred = (y_score >= threshold).astype(int)
        out_of_fold_scores.iloc[valid_idx] = y_score

        fold_rows.append(
            {
                "method": name,
                "fold": fold,
                "accuracy": accuracy_score(y_valid, y_pred),
                "precision": precision_score(y_valid, y_pred, zero_division=0),
                "recall": recall_score(y_valid, y_pred, zero_division=0),
                "specificity": specificity_score(y_valid, y_pred),
                "f1": f1_score(y_valid, y_pred, zero_division=0),
            }
        )

    return pd.DataFrame(fold_rows), out_of_fold_scores


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

fold_results = []
oof_scores = {}

for name, pipeline in pipelines.items():
    fold_df, scores = evaluate_pipeline(name, pipeline, X, y, cv)
    fold_results.append(fold_df)
    oof_scores[name] = scores

fold_results = pd.concat(fold_results, ignore_index=True)


## 17. Comparison table

The table reports mean cross-validation performance and fold-to-fold variability.


## Code: Summarize average performance

Compare recall and specificity together. Higher minority recall often comes with more majority false positives.


In [ ]:
#| echo: false
# Average fold-level metrics to compare strategies at the default 0.5 threshold.
summary = (
    fold_results.groupby("method")
    .agg(
        Accuracy=("accuracy", "mean"),
        Precision=("precision", "mean"),
        Recall=("recall", "mean"),
        Specificity=("specificity", "mean"),
        F1=("f1", "mean"),
        F1_std=("f1", "std"),
    )
    .sort_values("F1", ascending=False)
)

summary.round(3)


## Code: Inspect fold variability

Fold variability helps distinguish a consistently useful method from a lucky average.


In [ ]:
#| echo: false
# Inspect per-fold F1 to spot methods whose average hides instability.
fold_results.pivot(index="fold", columns="method", values="f1").round(3)


## 18. Confusion matrices

Here we use out-of-fold predictions. Each observation is predicted by a model that did not train on it.


## Code: Plot confusion matrices

Confusion matrices convert abstract metric changes into counts of mistakes.


In [ ]:
#| echo: false
# Convert out-of-fold probabilities into labels before drawing confusion matrices.
n_methods = len(oof_scores)
n_cols = 3
n_rows = int(np.ceil(n_methods / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3.5 * n_rows))
axes = np.asarray(axes).ravel()

for ax in axes[n_methods:]:
    ax.axis("off")

for ax, (name, scores) in zip(axes, oof_scores.items()):
    y_pred = (scores >= 0.5).astype(int)
    ConfusionMatrixDisplay.from_predictions(
        y,
        y_pred,
        labels=[0, 1],
        display_labels=["majority", "minority"],
        cmap="Blues",
        colorbar=False,
        ax=ax,
    )
    ax.set_title(name)

plt.tight_layout()
plt.show()


## 19. Precision-recall curves

A precision-recall curve shows how precision and recall change as we vary the classification threshold.

Most probabilistic classifiers first produce a positive-class score, then convert it into a label with a threshold.


## Reading precision-recall curves

- Lower thresholds predict more positives: recall usually rises and precision often falls.
- Higher thresholds predict fewer positives: precision may rise and recall often falls.

For imbalanced data, these curves focus directly on the rare positive class. The horizontal baseline is the positive-class rate.


## Code: Plot precision-recall curves

These curves use out-of-fold scores, so every score comes from a model that did not train on that row.


In [ ]:
#| echo: false
# Plot precision-recall curves from validation-only scores for each method.
fig, ax = plt.subplots(figsize=(8, 5))

for name, scores in oof_scores.items():
    PrecisionRecallDisplay.from_predictions(y, scores, name=name, ax=ax)

positive_rate = y.mean()
ax.axhline(positive_rate, color="black", linestyle="--", linewidth=1, label="positive rate")
ax.set_title("Out-of-fold precision-recall curves")
ax.legend(loc="best")
plt.show()


## 20. Threshold adjustment

`predict` usually applies a threshold of `0.5`. That threshold is not sacred: it is a decision rule, not a property of the model.

Changing the threshold changes the confusion matrix:

- A **lower threshold** produces more positive predictions. This usually reduces false negatives and increases recall, but it can increase false positives and lower precision.
- A **higher threshold** produces fewer positive predictions. This can reduce false positives and improve precision, but it can increase false negatives and lower recall.

In a high-cost false-negative scenario, such as fraud detection, medical screening, safety monitoring, or loan-default risk, we may prefer a lower threshold. The model will raise more alarms, but it misses fewer rare positive cases.

The right threshold depends on the operational cost of each error type. That is why threshold tuning should be evaluated with validation data, business constraints, and the metric that matches the real decision problem.


## Code: Evaluate thresholds

Thresholds turn probabilities into decisions. This table lets students choose a decision rule for a stated cost.


In [ ]:
# Recompute metrics over a grid of decision thresholds for one selected method.
def metrics_at_thresholds(y_true, y_score, thresholds):
    rows = []
    for threshold in thresholds:
        y_pred = (y_score >= threshold).astype(int)
        rows.append(
            {
                "threshold": threshold,
                "precision": precision_score(y_true, y_pred, zero_division=0),
                "recall": recall_score(y_true, y_pred, zero_division=0),
                "specificity": specificity_score(y_true, y_pred),
                "f1": f1_score(y_true, y_pred, zero_division=0),
            }
        )
    return pd.DataFrame(rows)


selected_method = "ADASYN"
threshold_grid = np.linspace(0.05, 0.95, 19)
threshold_results = metrics_at_thresholds(y, oof_scores[selected_method], threshold_grid)
threshold_results.round(3)


## Code: Plot the threshold trade-off

The plot makes the precision-recall trade-off easier to discuss than the table alone.


In [ ]:
#| echo: false
# Plot the metric trade-off created by moving the classification threshold.
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(threshold_results["threshold"], threshold_results["precision"], marker="o", label="precision")
ax.plot(threshold_results["threshold"], threshold_results["recall"], marker="o", label="recall")
ax.plot(threshold_results["threshold"], threshold_results["f1"], marker="o", label="f1")
ax.set_title(f"Threshold trade-off for {selected_method}")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend()
plt.show()


## 21. Choose a threshold for high-cost false negatives

Suppose a false negative costs 10 times as much as a false positive.

This is a simple teaching cost model, not a universal decision rule. In a real application, you would estimate costs with domain experts and validate the downstream effects.


## Code: Minimize threshold-sensitive cost

The cost model is intentionally simple; its purpose is to connect metrics to operational consequences.


In [ ]:
#| echo: false
# Search thresholds with a simple cost function where false negatives are expensive.
def cost_at_threshold(y_true, y_score, threshold, false_positive_cost=1, false_negative_cost=10):
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    total_cost = fp * false_positive_cost + fn * false_negative_cost
    return {
        "threshold": threshold,
        "false_positives": fp,
        "false_negatives": fn,
        "total_cost": total_cost,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


cost_rows = []
for method, scores in oof_scores.items():
    for threshold in np.linspace(0.05, 0.95, 91):
        row = cost_at_threshold(y, scores, threshold)
        row["method"] = method
        cost_rows.append(row)

cost_results = pd.DataFrame(cost_rows)
best_by_method = cost_results.loc[cost_results.groupby("method")["total_cost"].idxmin()]
best_by_method.sort_values("total_cost").round(3)


## 22. Leakage demonstration

The following example shows the wrong and right structure.

The **wrong** structure applies SMOTE before cross-validation. The validation folds are no longer clean because the synthetic data was produced using information from the full dataset.

The **right** structure places SMOTE inside an `imblearn` pipeline, so SMOTE is fitted only on each training fold.


## Code: Show a leakage pattern

This cell is intentionally unsafe first, then safe. Keep both results visible for the leakage discussion.


In [ ]:
# Contrast an intentionally leaky workflow with the leakage-safe pipeline workflow.
# Wrong pattern for teaching purposes only: do not use this as an evaluation design.
X_scaled = preprocessor.fit_transform(X)
X_leaky, y_leaky = SMOTE(random_state=RANDOM_STATE).fit_resample(X_scaled, y)

leaky_pipeline = Pipeline(
    [("classifier", logistic_regression())]
)

leaky_fold_results, _ = evaluate_pipeline(
    "Leaky SMOTE before CV",
    leaky_pipeline,
    pd.DataFrame(X_leaky),
    pd.Series(y_leaky),
    cv,
)

right_fold_results, _ = evaluate_pipeline("SMOTE inside CV", pipelines["SMOTE"], X, y, cv)

pd.concat([leaky_fold_results, right_fold_results])\
    .groupby("method")[["accuracy", "precision", "recall", "specificity", "f1"]]\
    .mean()\
    .round(3)


## 23. SMOTE-NC for mixed tabular data

`SMOTENC` is useful when predictors mix numerical and categorical columns.

The categorical feature indices must refer to the matrix seen by the sampler, so preprocessing order matters.


## SMOTE-NC pipeline note

A common design is to apply `SMOTENC` before one-hot encoding, or to use preprocessing that keeps categorical columns identifiable at the sampler step.

Be careful with plain SMOTE after one-hot encoding: interpolation can produce fractional category indicators.


## 24. Student challenge

Choose a pipeline for a high-cost false-negative scenario.

1. Define the cost of a false negative.
2. Select a primary metric.
3. Compare at least three imbalance strategies.
4. Tune the classification threshold.
5. Defend the final choice with evidence.


## Challenge datasets

Good options include:

- Credit Card Fraud Detection on Kaggle.
- Default of Credit Card Clients on UCI.
- Bank Marketing on UCI.
- Credit Approval on UCI.
- Telco Customer Churn on Kaggle.

Avoid optimizing only accuracy. Pick a metric that matches the scenario.


## References for imbalance methods

The BibTeX entries for the citation keys below are available in `refs/refs.bib`.

- SMOTE and SMOTE-NC: Chawla et al. [@chawla2002smote].
- Borderline-SMOTE: Han, Wang, and Mao [@han2005borderline].
- ADASYN: He et al. [@he2008adasyn].


## Important takeaway

**Resampling is part of model training, so it belongs inside the pipeline and inside cross-validation.**
